<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/content_authenticity_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re

# 공식 재난문자에서 실제로 반복 확인되는 문구 (2주차~4주차 실데이터에서 관찰)
OFFICIAL_PHRASE_MARKERS = [
    "안전에 유의", "대피", "출입금지", "우회", "통제", "자제",
    "확인바랍니다", "주의바랍니다", "발생", "예상", "발효",
]

# 스미싱 특유의 긴급성+보상미끼+클릭유도 조합 (실제 스미싱 샘플로 검증 필요 - 추정치)
SMISHING_RED_FLAGS = [
    "당첨", "무료", "쿠폰", "즉시 확인", "링크를 클릭", "인증번호",
    "환급", "대출", "저금리", "선착순", "본인확인", "계좌",
]

AGENCY_SUFFIXES = (
    "시청", "군청", "구청", "청", "통제소", "부", "본부", "센터",
    "시", "군", "구", "도", "경찰서", "소방서", "공사", "처",
)

AGENCY_TAG_PATTERN = re.compile(r'\[([^\[\]]+)\]\s*$')


def extract_agency_tag(message: str):
    """문장 끝 대괄호에서 발신 기관명을 뽑는다. 없으면 None."""
    if not isinstance(message, str):
        return None
    m = AGENCY_TAG_PATTERN.search(message.strip())
    return m.group(1).strip() if m else None


def content_authenticity_signals(message: str) -> dict:
    """
    문자 원문만으로 판단 가능한 신호들을 계산한다.
    DB 매칭 여부와 완전히 무관 - 텍스트 자체의 형식/어휘만 본다.
    """
    if not isinstance(message, str):
        message = ""

    agency = extract_agency_tag(message)
    agency_valid = bool(agency) and agency.endswith(AGENCY_SUFFIXES)

    official_marker_count = sum(p in message for p in OFFICIAL_PHRASE_MARKERS)
    red_flag_count = sum(p in message for p in SMISHING_RED_FLAGS)

    return {
        "agency_tag": agency,
        "agency_tag_present": agency is not None,
        "agency_pattern_valid": agency_valid,
        "official_marker_count": official_marker_count,
        "smishing_red_flag_count": red_flag_count,
    }


def content_authenticity_score(message: str) -> dict:
    """
    문자 원문만으로 낸 신뢰도 점수(0~1)와 근거.
    DB 매칭이 전혀 없을 때 disaster_reliability=0.5(중립) 대신
    이 점수로 대체하거나 블렌딩하는 용도.
    """
    sig = content_authenticity_signals(message)

    # 스미싱 신호가 하나라도 있으면 강하게 감점 (공식 문자에서 발견될 이유가 없는 문구들)
    if sig["smishing_red_flag_count"] > 0:
        score = max(0.0, 0.3 - 0.1 * sig["smishing_red_flag_count"])
        note = f"스미싱 의심 표현 {sig['smishing_red_flag_count']}개 발견 - 위험 신호"
        return {"score": round(score, 3), "signals": sig, "note": note}

    score = 0.5  # 기본값 - 정보부족과 동일한 출발점
    if sig["agency_tag_present"]:
        score += 0.15 if sig["agency_pattern_valid"] else -0.1
    score += min(sig["official_marker_count"] * 0.05, 0.15)

    score = max(0.0, min(1.0, score))

    note_parts = []
    if sig["agency_tag_present"]:
        note_parts.append(f"발신기관 태그 {'유효' if sig['agency_pattern_valid'] else '형식 이상'}({sig['agency_tag']})")
    else:
        note_parts.append("발신기관 태그 없음")
    note_parts.append(f"공식 문구 {sig['official_marker_count']}개 포함")

    return {"score": round(score, 3), "signals": sig, "note": " / ".join(note_parts)}


# ---------------------------------------------------------------------------
# 검증
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    test_messages = [
        # 실제 관찰된 정상 재난문자
        "오늘 05:50 미평천 청주시(장성2교)지점 심각단계. 하천범람에 대비 바랍니다. 내 위치, 침수우려지역 확인 vo.la/Vo27SU [금강홍수통제소]",
        "오늘 09:34 원주시 원동 한주아파트 101동 건물에서 화재 발생. 차량은 건물 주변 도로를 우회하고, 건물 내 시민은 건물 밖으로 대피하세요. [원주시]",
        # 발신기관 태그 없는 애매한 경우
        "화재가 발생했습니다 주의하세요",
        # 스미싱 스타일 예시(가상 - 실제 사례 아님, 추정 패턴 테스트용)
        "재난지원금 무료 쿠폰 당첨! 즉시 확인 후 인증번호 입력 http://bit.ly/abc123",
        "[국민재난안전처] 선착순 재난지원금 환급 대상자입니다. 계좌확인 필요 http://url.kr/xyz",
    ]

    for msg in test_messages:
        result = content_authenticity_score(msg)
        print(f"메시지: {msg[:50]}")
        print(f"  점수: {result['score']} | {result['note']}\n")


메시지: 오늘 05:50 미평천 청주시(장성2교)지점 심각단계. 하천범람에 대비 바랍니다. 내 위치
  점수: 0.7 | 발신기관 태그 유효(금강홍수통제소) / 공식 문구 1개 포함

메시지: 오늘 09:34 원주시 원동 한주아파트 101동 건물에서 화재 발생. 차량은 건물 주변 도
  점수: 0.8 | 발신기관 태그 유효(원주시) / 공식 문구 3개 포함

메시지: 화재가 발생했습니다 주의하세요
  점수: 0.55 | 발신기관 태그 없음 / 공식 문구 1개 포함

메시지: 재난지원금 무료 쿠폰 당첨! 즉시 확인 후 인증번호 입력 http://bit.ly/abc1
  점수: 0.0 | 스미싱 의심 표현 5개 발견 - 위험 신호

메시지: [국민재난안전처] 선착순 재난지원금 환급 대상자입니다. 계좌확인 필요 http://url.
  점수: 0.0 | 스미싱 의심 표현 3개 발견 - 위험 신호

